# An Orderbook Timeline Displayed in Jupyter Notebook
- October 2025, Alan Lenarcic

Note to really run this notebook you need a working Python environment with pyrus_kaiko`` package installed and the gpuorderbook``` functionality compiled with ES builder.

See README for gpuorderbook installation.

We hope to be able to compile this Notebook to HTML using nbconvert

Here we use anywidget to render javascript inside a jupyter notebook. We use ```pyrus_kaiko`` package to simulate orderbook data.
nbconvert will be required to export as htm
To try from command line jupyter nbconvert --to html --execute Demo_GPUOB.ipynb

## Not quite sure on anywidget functionality
The HTML version of this notebook definitely renders some aspects of the GPU notebook when opened separately in Chrome (outside of jupyter server). That said the render is not entirely clean.

In [ ]:
from pyrus_kaiko import pyrus_kaiko
## Note that pyo3 has changeable module inclusion adivce
## Here we use a "from pyrus_package import pyrus_package" interface which runs our "__init__.py" script

In [ ]:
import os; 
from pyrus_kaiko import pyrus_kaiko;
from pyrus_kaiko import ob;

nGen=20000; #nGen = 5000; 
nMidPoints = 4000;
import numpy as np; import pandas as pd;
import pyarrow as pa;
out_t, nr, v_d_fake, bprices, sprices, \
  write_sum_q, write_sum_pq, write_avp, write_worstpi, \
  write_nlevels, write_atq, \
  v_w, verbose, time_minmax, a_Fake_OB, MidTable = \
  ob.fake_for_algo(nGen = nGen,nMidPoints=nMidPoints,AddMidPrice=True, deltaSec=.01, v_bhit=False,fast_check="PARALLEL");

In [ ]:
# fake NBBO from the fake Midpoint
import copy
pd_dem = copy.copy(MidTable);
pd_dem = pd_dem[(pd_dem.time <= time_minmax[1]) & (pd_dem.time.shift(-1,fill_value=time_minmax[1])  >= time_minmax[0])].reset_index(inplace=False, drop=True)
rtt = pd_dem['time'].copy().to_numpy(); rtt[rtt < time_minmax[0]] = time_minmax[0];
pd_dem['time'] = rtt;
pd_dem['best_bid'] = pd_dem['nbm'] - .01;
pd_dem['best_ask'] = pd_dem['nbm'] + .01;
pd_dem['b_sum_q_1'] = 100;
pd_dem['s_sum_q_1'] = 100;

In [ ]:
nTrades = 200; trade_i = np.asarray(np.unique(np.random.choice(np.arange(len(pd_dem)), nTrades, replace=False)))
ts = np.random.choice(np.asarray([0,1]),nTrades)
trades =  pd_dem.iloc[trade_i].reset_index(inplace=False, drop=True);
tprice = trades['best_bid'].to_numpy();
tprice[ts == 0] == trades['best_ask'].to_numpy()[ts == 0]
tqty = trades['b_sum_q_1'];  tqty[ts==0] == trades['s_sum_q_1'][ts==0];
trades = pd.DataFrame({'time':trades['time'],'price':tprice,'qty': (-2*ts + 1) * tqty})
trades['time'] = (trades['time']-pd_dem['time'][0]).astype(np.int64)/1000000000.0

trades

In [ ]:
dt = str(time_minmax[0])[0:10]; st_time =  str(time_minmax[0])[11:26]; unit = 0;
time0 = time_minmax[0];
nbbo = pd_dem[['time','best_bid','best_ask']].reset_index(inplace=False, drop=True); nbbo.columns = ['time','nbb','nbo']; nbbo['time'] = ((nbbo['time'] - time0).astype(np.int64)) / 1000000000.0
tmin = np.min(nbbo['time']); tmax = np.max(nbbo['time']);
inner_pmin = np.min(nbbo['nbb']);  inner_pmax = np.max(nbbo['nbo']);
pmin = inner_pmin - 1 * (inner_pmax - inner_pmin);  pmax = inner_pmax + 1 * (inner_pmax - inner_pmin);
height = 400; width = 800;
lwd_h = .01; lwd_w = (400/800) * lwd_h;
max_qty = np.max(out_t['cm_qty']); pow_qty = .25;
trade_mul_fac=.1; msg_mul_fac = .1;
buys = a_Fake_OB[a_Fake_OB['bs'] == b'b'][['price','qty','t0','t1']].reset_index(inplace=False, drop=True); buys['t0'] = (buys['t0'].astype('datetime64[ns]') - time0).astype(np.int64) / 1000000000.0;  buys['t1'] = (buys['t1'].astype('datetime64[ns]') - time0).astype(np.int64) / 1000000000.0;
buys.columns = ['price', 'qty', 'open','close']
buys

sells = a_Fake_OB[a_Fake_OB['bs'] == b's'][['price','qty','t0','t1']].reset_index(inplace=False, drop=True); sells['t0'] = (sells['t0'].astype('datetime64[ns]') - time0).astype(np.int64) / 1000000000.0;  sells['t1'] = (sells['t1'].astype('datetime64[ns]') - time0).astype(np.int64) / 1000000000.0;
sells.columns = ['price', 'qty', 'open','close']
sells
data_type = 'generated_data';

In [ ]:
out_t
mxtime = np.max(out_t['time']);
buys = out_t[out_t['side'] == b'b'][['price','cm_qty','time']].reset_index(inplace=False,drop=True);
close = buys['time'].shift(-1,fill_value=mxtime);
close[buys['price'] != buys['price'].shift(-1,fill_value=-1)] = mxtime; buys['close'] = close;  buys.columns = ['price','qty','open','close'];  
buys['open'] = (buys['open'].astype('datetime64[ns]') - time0).astype(np.int64) / 1000000000.0;  
buys['close'] = (buys['close'].astype('datetime64[ns]') - time0).astype(np.int64) / 1000000000.0;
buys = buys[buys.qty > 0].reset_index(inplace=False, drop=True)

sells = out_t[out_t['side'] == b's'][['price','cm_qty','time']].reset_index(inplace=False,drop=True);
close = sells['time'].shift(-1,fill_value=mxtime);
close[sells['price'] != sells['price'].shift(-1,fill_value=-1)] = mxtime; sells['close'] = close;  sells.columns = ['price','qty','open','close'];  
sells['open'] = (sells['open'].astype('datetime64[ns]') - time0).astype(np.int64) / 1000000000.0;  
sells['close'] = (sells['close'].astype('datetime64[ns]') - time0).astype(np.int64) / 1000000000.0;
sells = sells[sells.qty > 0].reset_index(inplace=False, drop=True);
sells

In [ ]:
test_data = { "dt": dt, 'st_time':st_time, 'unit':unit, 'tmin':tmin, 'tmax':tmax, 'pmin':pmin, 'pmax':pmax,'height':height, 'width':width,
                'lwd_h':lwd_h, 'lwd_w':lwd_w, 'max_qty':max_qty, 'pow_qty':pow_qty,'trade_mul_fac':trade_mul_fac, 'msg_mul_fac':msg_mul_fac,
                'nbbo':nbbo.to_dict(orient='list'),'buys':buys.to_dict(orient='list'),'sells':sells.to_dict(orient='list'),'trades':trades.to_dict(orient='list'),'data_type':'generated_data'};
import json;
aD = json.dumps(test_data)

In [ ]:
import pathlib
import anywidget
import traitlets
from IPython.display import display, HTML
display(HTML("<style>.container { width:100% ! important; }</style>"))


#  NumFilled:0,
#  NumAll:1,
#  NumShares:2,
#  NumPennies:3,
#  MultipleSpread:4,
#  TotalDollars:5,
#  PctDepth:6,
#  ExpDecay:7,
#  Unknown:-1

from IPython.display import HTML, display
#html_code = "<h1>Hello, world!</h1><p>This is a paragraph.</p>"
#install_div_space = \
#"""<div id="gl_div" name="gl_div"> <div>"""
#display(HTML(install_div_space))
kalgo = 5; v_d = [100000000,500000000,1000000000]
#kalgo = 2; v_d = [10000,50000,100000];
kalgo = 3; v_d = [40,80, 120]
w_sum_q = 1; w_sum_pq = 1; w_avp = 1; w_wp = 1; w_nocc = 0; w_atp = 0; w_crit_pi = 1; w_crit_p = 1;
verbose = 2;
class obwidget(anywidget.AnyWidget):
    _esm = "./outwidget/static/widget_index.js"
    _css = "./outwidget/static/widget_index.css"
    data = traitlets.Dict(test_data).tag(sync=True)
    value = traitlets.Int(0).tag(sync=True)
    kalgo = traitlets.Int(kalgo).tag(sync=True)
    v_d = traitlets.List(traitlets.Float(), default_value=v_d).tag(sync=True)
    w_sum_q = traitlets.Int(w_sum_q).tag(sync=True)
    w_sum_pq = traitlets.Int(w_sum_pq).tag(sync=True)
    w_avp = traitlets.Int(w_avp).tag(sync=True)
    w_wp = traitlets.Int(w_wp).tag(sync=True)
    w_nocc = traitlets.Int(w_nocc).tag(sync=True)
    w_atp = traitlets.Int(w_atp).tag(sync=True)
    w_crit_pi = traitlets.Int(w_crit_pi).tag(sync=True)
    w_crit_p = traitlets.Int(w_crit_p).tag(sync=True)
    verbose = traitlets.Int(verbose).tag(sync=True)
    

obwidget()

In [ ]:
#UNCOMMENT_FOR_TEST_WIDGET = """
import pathlib
import anywidget
import traitlets
from IPython.display import display, HTML
display(HTML("<style>.container { width:100% ! important; }</style>"))


from IPython.display import HTML, display
html_code = "<h1>Hello, world!</h1><p>This is a paragraph.</p>"
install_div_space = "<div id=\"gl_div\" name=\"gl_div\"> <div>"
display(HTML(install_div_space))

import copy;
v_d = [1.0,2.0,3.0]; kalgo = 0;
w_sum_q = 1; w_sum_pq = 1; w_avp = 0; w_wp = 1; w_nocc = 0; w_atp = 0;
verbose = 2;
class obwidget(anywidget.AnyWidget):
    _esm = "./outwidget/static/widget_index.js"
    _css = "./outwidget/static/widget_index.css"
    value = traitlets.Int(0).tag(sync=True)
    kalgo = traitlets.Int(kalgo).tag(sync=True)
    v_d = traitlets.List(traitlets.Float(), default_value=copy.copy(v_d)).tag(sync=True)
    #v_fd = traitlets.List(traitlets.Float(), default_value=copy.copy(v_d)).tag(sync=True)
    w_sum_q = traitlets.Int(w_sum_q).tag(sync=True)
    w_sum_pq = traitlets.Int(w_sum_pq).tag(sync=True)
    w_avp = traitlets.Int(w_avp).tag(sync=True)
    w_wp = traitlets.Int(w_wp).tag(sync=True)
    w_nocc = traitlets.Int(w_nocc).tag(sync=True)
    w_atp = traitlets.Int(w_atp).tag(sync=True)
    verbose = traitlets.Int(verbose).tag(sync=True)

obwidget()